<div align="right"><a href="S14-Quantization-of-filter-coefficients.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L14-Quantisierung-der-Filterkoeffizienten.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L13 Entwurf rekursiver Filter](./L13-Entwurf-rekursiver-Filter.ipynb) | [Aufgabe 14](./A14-Quantisierung-der-Filterkoeffizienten.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L15 Fensterfunktionen](./L15-Fensterfunktionen.ipynb)

</div>

---


# Quantisierung von Filterkoeffizienten

In Abschnitt 4.4.1 wurde während der Analyse der **Parameterquantisierung** gezeigt, dass die Pole empfindlicher werden, je mehr Pole vorhanden sind und je näher sie zueinander liegen. Bei den Nullstellen ist ein ähnliches Verhalten zu beobachten. Daher ist die **dritte kanonische Form** (Übung 4.4.1) von großer Bedeutung für die Realisierung höherer Ordnung von Systemen.

Diese Aufgabe zeigt ein Beispiel, wie die **Quantisierung von Filterkoeffizienten** die Systemfunktion $H(z)$ beeinflusst. Die Funktion `lfxquant()` implementiert einen linearen Quantisierer:

$$
-1 \leq [v]_Q \leq 1 - 2^{-l+1},
$$

wobei $l$ die Anzahl der Bits ist und $v$ die zu quantisierende Zahl darstellt. Um die Koeffizienten eines digitalen Filters zu quantisieren, müssen die Koeffizienten **normalisiert** werden. Die Funktion `[aq, n] = lcoefrnd(a, l)` berechnet den Vektor der quantisierten Koeffizienten `aq` sowie den Normalisierungsfaktor `n`.

---

### a)

Für unsere Analyse verwenden wir einen **Cauer-Tiefpassfilter** 7. Ordnung mit den folgenden Spezifikationen:

* $3,\text{dB}$-Eckfrequenz: $0.4 \pi$
* Durchlassbereichsdämpfung: $R_P = 0.1,\text{dB}$
* Sperrbereichsdämpfung: $R_S = 40,\text{dB}$

In Python (z. B. mit `scipy.signal.ellip`) kann man die Koeffizienten berechnen:

```python
from scipy.signal import ellip
b, a = ellip(7, 0.1, 40, 0.4, btype='low', output='ba')
```

Plotten Sie den Frequenzgang mit Python (z. B. `scipy.signal.freqz` und `matplotlib`).

---

### b)

Quantisieren Sie die Filterkoeffizienten mit **8 Bits**, z. B.:

```python
bq, nb = lcoefrnd(b, 8)
aq, na = lcoefrnd(a, 8)
```

Dies ergibt die quantisierten Koeffizienten `bq` und `aq` zusammen mit den Normalisierungsfaktoren `nb` und `na`.

Plotten Sie die resultierenden Übertragungsfunktionen sowie die Version mit nicht quantisierten Koeffizienten (Übung 4.4.1.a).

**Hinweis:** Um die Übertragungsfunktion des Filters mit quantisierten Parametern zu plotten, muss die Normalisierung der Koeffizienten invertiert werden:

```python
from scipy.signal import freqz
import matplotlib.pyplot as plt

w, h = freqz(bq, aq, 512)
Hq = nb/na * h
plt.plot(w, 20 * np.log10(abs(Hq)))
plt.show()
```

---

### c)

Bei Anwendung der **8-Bit-Quantisierung** ergibt sich eine große Abweichung der Übertragungsfunktion im Vergleich zum vorgegebenen **Toleranzschema**. Die zulässigen Abweichungen sind:

* $R_D^+ = 0.15,\text{dB}$
* $R_D^- = -0.15,\text{dB}$
* $R_S = -38,\text{dB}$

> $R_D^+$ und $R_D^-$ bezeichnen die maximalen Schwankungen im Durchlassbereich, wobei $R_D^- = R_P$.

Bestimmen Sie die **minimale Wortlänge** der Filterkoeffizienten, um dieses Toleranzschema einzuhalten.

---

Es gibt verschiedene Strukturen digitaler Filter. Die **Direktform** wird in Python mit `scipy.signal.lfilter` implementiert, die bereits zuvor verwendet wurde.

Um die Effekte der Quantisierung zu reduzieren, wird meist die **dritte kanonische Form** verwendet, bei der das System aus mehreren erst- und zweitordentlichen Filtern in Serie realisiert wird.

Im Folgenden analysieren wir die Effekte der **Parameterquantisierung**, wenn die **dritte kanonische Form** für den oberen Cauer-Tiefpass eingesetzt wird.

---

### d)

Berechnen Sie die Filterkoeffizienten der **zweiten Ordnung Sektionen (SOS)** des Cauer-Tiefpasses mit `lcascade()` und die resultierende Systemfunktion:

$$
H(z) = \prod_{i=1}^{p} H_i(z)
$$

Quantisieren Sie die Filterkoeffizienten der SOS mit `lcoefrnd()` für Wortlängen $l = 8, 10, 12, \dots$ Bits und berechnen Sie die Übertragungsfunktion der Kaskade mit quantisierten Koeffizienten.

Bestimmen Sie, wie viele Bits notwendig sind, um die Filterkoeffizienten der SOS zu quantisieren und das **Toleranzschema** (Übung 4.4.1.a) einzuhalten. Vergleichen Sie diese Wortlänge mit der minimalen Wortlänge, die für die **direkte Implementierung** des Filters nötig war, um das Toleranzschema zu erfüllen (Übung 4.4.1.c).

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ellip, freqz
from helper_functions import lcoefrnd, lcascade

# -------------------------------------------------------------------
# Task a: Design Cauer Lowpass
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)

plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H)))
plt.grid()
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')
plt.title('Cauer Lowpass')


# -------------------------------------------------------------------
# Task b: Quantize filter coefficients
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)

bq, nb = lcoefrnd(b, 8)
aq, na = lcoefrnd(a, 8)
Wq, Hq_temp = freqz(bq, aq, 512)
Hq = (nb / na) * Hq_temp

plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H)))
plt.plot(W / np.pi, 20 * np.log10(np.abs(Hq)), '-.')
plt.grid()
plt.title('- Unquantized Coefficients   -. 8-bit Quantized Coefficients')
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')


# -------------------------------------------------------------------
# Task c: Determine minimum word length
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)
v_length = [8, 12, 16]

for l in v_length:
    bq, nb = lcoefrnd(b, l)
    aq, na = lcoefrnd(a, l)

    _, Hq_temp = freqz(bq, aq, 512)
    Hq = (nb / na) * Hq_temp

    # Passband
    plt.figure()
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(H[0:208])))
    plt.axis([0, 0.5, -1.4, 0.3])
    plt.grid()
    plt.plot(W[0:208] / np.pi, -0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, +0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(Hq[0:208])), '--')
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'Passband: - Unquantized, -- Quantized with {l} Bits')

    # Stopband
    plt.figure()
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(H[214:512])))
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(Hq[214:512])), '--')
    plt.plot(W[214:512] / np.pi, -38 * np.ones(512 - 214), 'w')
    plt.grid()
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'Stopband: - Unquantized, -- Quantized with {l} Bits')


# -------------------------------------------------------------------
# Task d: 3rd canonical form quantization
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
B, A = lcascade(b, a)

m, n = B.shape
Hi = np.zeros((512, m), dtype=complex)

for k in range(m):
    _, temp = freqz(B[k, :], A[k, :], 512)
    Hi[:, k] = temp

H3k = np.prod(Hi, axis=1)

# Only for 12 bits, like MATLAB
v_length = [12]

for l in v_length:
    Bq = np.zeros_like(B)
    Aq = np.zeros_like(A)
    nb = np.zeros(m)
    na = np.zeros(m)

    for k in range(m):
        Bq[k, :], nb[k] = lcoefrnd(B[k, :], l)
        Aq[k, :], na[k] = lcoefrnd(A[k, :], l)

    Hq = np.zeros((512, m), dtype=complex)
    for k in range(m):
        _, temp = freqz(Bq[k, :], Aq[k, :], 512)
        Hq[:, k] = (nb[k] / na[k]) * temp

    Hqg = np.prod(Hq, axis=1)

    # Passband
    plt.figure()
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(H3k[0:208])))
    plt.axis([0, 0.5, -1.4, 0.3])
    plt.grid()
    plt.plot(W[0:208] / np.pi, -0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, +0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(Hqg[0:208])), '--')
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'3rd Canonical: Pass.: - Unquantized, -- Quantized with {l} Bits')

    # Stopband
    plt.figure()
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(H3k[214:512])))
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(Hqg[214:512])), '--')
    plt.plot(W[214:512] / np.pi, -38 * np.ones(512 - 214), 'w')
    plt.grid()
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'3rd Canonical: Stop.: - Unquantized, -- Quantized with {l} Bits')


plt.show()
# END SOLUTION

## Lösung: Quantisierung von Filterkoeffizienten

### a)-b)

Die Abbildung zeigt den **Frequenzgang** des Cauer-Tiefpassfilters.

<img src="./pngs/ldsv4a04_ab.png" width="250">

Wenn die Koeffizienten beider Polynome (Zähler und Nenner) gemäß

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1}
$$

mit 8 Bits quantisiert werden, ergibt sich die dargestellte Amplitudencharakteristik. Ein Vergleich der beiden Funktionen zeigt eine deutliche Variation im Übergang vom **Durchlass-** zum **Sperrbereich**. Das vorgegebene Toleranzschema wird vom System mit quantisierten Koeffizienten **nicht erfüllt**.

---

### c)

Unter Verwendung der Toleranzabweichungen $R_D^+$ und $R_D^-$ aus der Aufgabe werden für die Realisierung in **Direktform** als Einzel-System mindestens **16-Bit-Worte** benötigt.

Die folgenden Abbildungen zeigen den Durchlass- und Sperrbereich der quantisierten Übertragungsfunktion im Vergleich zur nicht quantisierten Version:

<img src="./pngs/ldsv4a04_c1.png" width="250">
<img src="./pngs/ldsv4a04_c2.png" width="250">

Bei einer Wortlänge von 16 Bit gibt es eine **kleine Abweichung** vor der abfallenden Kante im Durchlassbereich, während im Sperrbereich keine Variation auftritt.

---

### d)

Wenn der Cauer-Tiefpass (Aufgabenteil a) in **dritter kanonischer Form** realisiert wird und die **minimale Wortlänge** für die Quantisierung bestimmt wird, um das Toleranzschema einzuhalten, zeigt sich, dass eine **12-Bit-Quantisierung** ausreichend ist.

Im Vergleich zur direkten Implementierung (Aufgabenteil b) werden somit **4 Bits weniger** benötigt. Die Abbildungen zeigen die Amplitudencharakteristiken im **Sperr-** und **Durchlassbereich**:

<img src="./pngs/ldsv4a04_d1.png" width="250">
<img src="./pngs/ldsv4a04_d2.png" width="250">